# 第4回：Python超入門(2) 解答例

> [!NOTE]
> 解答はあくまで「一例」です。同じ結果が出れば、別の書き方でも正解です。

---

## 演習 1-1：辞書から DataFrame を作る


In [ ]:
import pandas as pd

data = {
    "科目":  ["数学", "英語", "Python"],
    "点数":  [75, 55, 90],
    "合格":  [True, False, True]
}

df_scores = pd.DataFrame(data)
print(df_scores)


---

## 演習 1-2：CSV に保存して読み込む


In [ ]:
# 保存
df_scores.to_csv("my_scores.csv", index=False)

# 読み込んで確認
df_loaded = pd.read_csv("my_scores.csv")
print(df_loaded)


---

## 演習 2-1：shape・columns・dtypes の確認


In [ ]:
print(df.shape)
print(df.columns)
print(df.dtypes)


**答え：** `(17000, 9)` → 17000行・9列。すべての列が `float64`（小数型）。

---

## 演習 2-2：3列を取り出す


In [ ]:
subset = df[["longitude", "latitude", "median_house_value"]]
subset.head()


---

## 演習 3-1：median_income の統計量


In [ ]:
col = "median_income"
print("平均:", df[col].mean())
print("中央値:", df[col].median())
print("標準偏差:", df[col].std())
print("Q1:", df[col].quantile(0.25))
print("Q3:", df[col].quantile(0.75))


**考察の例：**  
収入は住宅価格より平均と中央値の差が小さい傾向がある。住宅価格は上限（500000ドル）付近に外れ値のようなデータが集中しており、平均が中央値より大きくなりやすい（右に歪んだ分布）。

---

## 演習 3-2：housing_median_age の標準偏差を読み取る


In [ ]:
df.describe()


`housing_median_age` の `mean`（平均）が約28〜29年程度、`std`（標準偏差）が約12年程度であれば、データが平均の前後かなり広い範囲にばらついていることを意味する。

---

## 演習 4-1：築年数30年以上を絞り込む


In [ ]:
df_old = df[df["housing_median_age"] >= 30]
print(df_old.shape)


---

## 演習 4-2：価格200000以下かつ収入3以上


In [ ]:
affordable = df[(df["median_house_value"] <= 200000) & (df["median_income"] >= 3)]
print(affordable.shape)


---

## 演習 5-1：price_to_income 列の追加と統計


In [ ]:
df["price_to_income"] = df["median_house_value"] / df["median_income"]

print("平均:", df["price_to_income"].mean())
print("中央値:", df["price_to_income"].median())


値が大きいほど「収入に対して住宅価格が高い」地域であることを示す。

---

## 演習 5-2：築年数ごとの収入の agg 集計


In [ ]:
result = df.groupby("housing_median_age")["median_income"].agg(["mean", "std", "count"])
result.columns = ["平均収入", "標準偏差", "件数"]
result.head(10)


---

## 演習 6-1：Pandas 組み込みグラフでヒストグラム


In [ ]:
import matplotlib.pyplot as plt

df["median_income"].plot.hist(bins=30, title="収入の分布", figsize=(8, 4))
plt.show()


---

## 演習 6-2：築年数ごとの平均価格を棒グラフで表示


In [ ]:
age_price = df.groupby("housing_median_age")["median_house_value"].mean()
age_price.plot.bar(figsize=(14, 5), title="築年数ごとの平均住宅価格", color="steelblue")
plt.xlabel("築年数")
plt.ylabel("平均価格（ドル）")
plt.show()


---

## 演習 7-1：total_rooms のヒストグラム


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df["total_rooms"], bins=40, color="mediumpurple", edgecolor="white")
plt.title("総部屋数の分布")
plt.xlabel("総部屋数")
plt.ylabel("件数")
plt.show()


---

## 演習 7-2：折れ線 vs 棒グラフの選択

**正解：どちらも一定の根拠があるが、「築年数」は連続した順序データなので折れ線グラフが適切なことが多い。**  
棒グラフはカテゴリが多いと見づらくなる。ただし「カテゴリとして比較したい」なら棒グラフも可。


In [ ]:
age_price = df.groupby("housing_median_age")["median_house_value"].mean()

plt.figure(figsize=(12, 5))
plt.plot(age_price.index, age_price.values,
         color="navy", linewidth=2, marker="o", markersize=4)
plt.title("築年数ごとの平均住宅価格（折れ線グラフ）")
plt.xlabel("築年数")
plt.ylabel("平均価格（ドル）")
plt.grid(True, alpha=0.3)
plt.show()


---

## 演習 8-1：横2つのヒストグラム


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(df["median_income"], bins=30, color="steelblue")
axes[0].set_title("収入の分布")
axes[0].set_xlabel("収入")
axes[0].set_ylabel("件数")

axes[1].hist(df["median_house_value"], bins=30, color="coral")
axes[1].set_title("住宅価格の分布")
axes[1].set_xlabel("価格（ドル）")
axes[1].set_ylabel("件数")

plt.tight_layout()
plt.show()


---

## 演習 8-2：total_rooms の箱ひげ図


In [ ]:
plt.figure(figsize=(8, 6))
plt.boxplot(df["total_rooms"])
plt.title("総部屋数の箱ひげ図")
plt.ylabel("総部屋数")
plt.show()


**観察：** `total_rooms` は右側に多数の外れ値（○で表示される点）が見られる。これは一部に非常に部屋数が多い集合住宅（アパート等）が含まれていることを示している。外れ値が多い場合は**ノンパラメトリック手法（中央値・四分位数）**で分析するのが適切。

---

## 演習 9-1：Seaborn の histplot（KDE付き）


In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="median_income", bins=30, kde=True, color="coral")
plt.title("収入の分布（KDE付き）")
plt.show()


**Matplotlibとの違い：** 背景がグリッド付き白色になり、KDE曲線（分布の形を滑らかに表した曲線）が重ねて表示されるため、データのピークや偏りがより直感的にわかる。

---

## 演習 9-2：テーマを変えてグラフを描く


In [ ]:
sns.set_theme(style="darkgrid")  # ← テーマを変える

plt.figure(figsize=(10, 5))
sns.histplot(data=df, x="median_income", bins=30, kde=True, color="coral")
plt.title("収入の分布（darkgridテーマ）")
plt.show()


---

## 演習 10-1：相関行列から読み取る


In [ ]:
corr_matrix = df.corr(numeric_only=True)

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1)
plt.title("相関行列")
plt.show()


**答え：**
1. `median_house_value`（価格）と最も相関が高い変数：`median_income`（収入）、相関係数約 **0.69**
2. 負の相関が最も強いペア：`longitude`（経度）と `latitude`（緯度）など（地理的な特性によるもの）

---

## 演習 10-2：築年数グループごとの収入の箱ひげ図


In [ ]:
# age_group 列がない場合は作成
if "age_group" not in df.columns:
    df["age_group"] = (df["housing_median_age"] // 10 * 10).astype(int)

plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="age_group", y="median_income", palette="Greens")
plt.title("築年数グループごとの収入分布")
plt.xlabel("築年数グループ（年代）")
plt.ylabel("収入（万ドル）")
plt.show()


**観察の例：** 明確な傾向よりも各グループのばらつきが大きく、築年数と収入には強い関係がない可能性がある（相関行列でも確認できる）。
